# Lab 6: Word Sense Disambiguation

In [1]:
from pathlib import Path
import tarfile

paths = [Path('../sts/resources/test-gold.tgz'), Path('sts/resources/test-gold.tgz'), Path('test-gold.tgz')]
path = next((p for p in paths if p.is_file()), None)
if path is None:
    raise FileNotFoundError()

with tarfile.open(path) as tar:
    pairs = [line.split('\t') for line in tar.extractfile('test-gold/STS.input.SMTeuroparl.txt').read().decode('utf-8').splitlines()]
    gs = [float(line) for line in tar.extractfile('test-gold/STS.gs.SMTeuroparl.txt').read().decode('utf-8').splitlines()]

assert len(pairs) == len(gs)
assert all(len(pair) == 2 for pair in pairs)
len(pairs), len(gs)

(459, 459)

In [2]:
print(f"{'#':<3} {'sentence 1':<65} sentence 2")
for i, (a, b) in enumerate(pairs[:5], 1):
    print(f"{i:<3} {a[:62]:<65} {b[:62]}")

#   sentence 1                                                        sentence 2
1   The leaders have now been given a new chance and let us hope t    The leaders benefit aujourd' hui of a new luck and let's let t
2   Amendment No 7 proposes certain changes in the references to p    Amendment No 7 is proposing certain changes in the references 
3   Let me remind you that our allies include fervent supporters o    I would like to remind you that among our allies, there are st
4   The vote will take place today at 5.30 p.m.                       The vote will take place at 5.30pm
5   The fishermen are inactive, tired and disappointed.               The fishermen are inactive, tired and disappointed.


## Words

In [3]:
import re
from nltk.metrics import jaccard_distance

unique_sentences = list(dict.fromkeys(sentence for pair in pairs for sentence in pair))
word_pattern = re.compile(r'\b\w+\b')
tokens = {sentence: word_pattern.findall(sentence) for sentence in unique_sentences}
word_sets = {sentence: {word.casefold() for word in words} for sentence, words in tokens.items()}

print(f'{len(pairs)} pairs; {2 * len(pairs)} sentence occurrences; {len(unique_sentences)} unique sentences')

459 pairs; 918 sentence occurrences; 338 unique sentences


## Lemmas

In [4]:
import nltk
from functools import cache
from nltk.stem import WordNetLemmatizer

resources = {
    'averaged_perceptron_tagger_eng': 'taggers/averaged_perceptron_tagger_eng/',
    'wordnet': 'corpora/wordnet/',
}
for package, resource in resources.items():
    try:
        nltk.data.find(resource)
    except LookupError:
        nltk.download(package, quiet=True, raise_on_error=True)

lemmatizer = WordNetLemmatizer()
wordnet_pos = {'N': 'n', 'V': 'v', 'J': 'a', 'R': 'r'}

@cache
def lemma(word, pos):
    return lemmatizer.lemmatize(word, pos) if pos else word

In [5]:
tagged = dict(zip(unique_sentences, nltk.pos_tag_sents(list(tokens.values()))))
lemma_sets = {
    sentence: {lemma(word.casefold(), wordnet_pos.get(tag[0])) for word, tag in tagged_sentence}
    for sentence, tagged_sentence in tagged.items()
}

## Senses (Lesk)

Lesk chooses the sense whose definition shares more words with the sentence. We use the whole sentence as context and the POS tag to filter the candidate synsets (for adjectives we also include the satellite ones, `s`).

In [6]:
from nltk.corpus import wordnet as wn
from nltk.wsd import lesk

context = ['I', 'went', 'to', 'the', 'bank', 'to', 'deposit', 'money', '.']
synset = lesk(context, 'bank', 'n')
synset.name(), synset.definition()

('depository_financial_institution.n.01',
 'a financial institution that accepts deposits and channels the money into lending activities')

In [7]:
def sense(context, word, pos):
    synsets = wn.synsets(word, pos)
    return lesk(context, word, synsets=synsets) if synsets else None

sense_sets, sense_lemma_sets = {}, {}
disambiguated = candidates = 0
for sentence, tagged_sentence in tagged.items():
    context = [word.casefold() for word, _ in tagged_sentence]
    senses, mixed = set(), set()
    for word, tag in tagged_sentence:
        word, pos = word.casefold(), wordnet_pos.get(tag[0])
        synset = sense(context, word, pos) if pos else None
        candidates += pos is not None
        if synset:
            disambiguated += 1
            senses.add(synset.name())
            mixed.add(synset.name())
        else:
            mixed.add(lemma(word, pos))
    sense_sets[sentence], sense_lemma_sets[sentence] = senses, mixed

print(f'{disambiguated} of {candidates} open-class tokens got a sense')
print(f'{sum(not s for s in sense_sets.values())} sentences with no senses')

2130 of 2279 open-class tokens got a sense
0 sentences with no senses


In [8]:
for a, b in pairs[:2]:
    print(a, sorted(sense_sets[a]), sep='\n  ')
    print(b, sorted(sense_sets[b]), sep='\n  ')
    print()

The leaders have now been given a new chance and let us hope they seize it.
  ['assume.v.06', 'be.v.01', 'fresh.s.03', 'give.v.01', 'have.v.01', 'hope.v.01', 'leadership.n.02', 'let.v.01', 'now.r.01', 'probability.n.01']
The leaders benefit aujourd' hui of a new luck and let's let them therefore seize it.
  ['fresh.s.03', 'impound.v.01', 'leadership.n.02', 'let.v.01', 'luck.n.03', 'profit.v.01', 'therefore.r.01']

Amendment No 7 proposes certain changes in the references to paragraphs.
  ['amendment.n.01', 'certain.a.04', 'change.n.01', 'character.n.07', 'nobelium.n.01', 'nominate.v.02', 'paragraph.v.02']
Amendment No 7 is proposing certain changes in the references to paragraphs.
  ['amendment.n.02', 'be.v.12', 'certain.a.04', 'change.n.05', 'character.n.07', 'nobelium.n.01', 'nominate.v.02', 'paragraph.v.02']



## Senses (UKB)

UKB uses the WordNet graph to choose the senses of all the words of the sentence together. We call it through TextServer.

In [ ]:
import json
import sys
import requests

sys.path.insert(0, str(Path('../sts/resources').resolve()))
from textserver import TextServer

def read_env(paths=(Path('../.env'), Path('.env'))):
    path = next((p for p in paths if p.is_file()), None)
    if path is None:
        raise FileNotFoundError('.env with TS_USER and TS_PASSWORD not found')
    lines = (line.split('=', 1) for line in path.read_text().splitlines() if '=' in line and not line.lstrip().startswith('#'))
    return {key.strip(): value.strip().strip('\'"') for key, value in lines}

env = read_env()
ts = TextServer(env['TS_USER'], env['TS_PASSWORD'], 'senses')

def token_info(token):
    return [token['form'], token['lemma'], token['tag'], token['pos'], token.get('wn', 'N/A')]

def ukb_senses(batch):
    ts.service = 'senses'
    paragraphs = json.loads(ts.query('\n\n'.join(batch)))['paragraphs']
    if len(batch) == 1:
        paragraphs = [{'sentences': [s for p in paragraphs for s in p['sentences']]}]
    if len(paragraphs) != len(batch):
        return None
    return [[token_info(t) for s in p['sentences'] for t in s['tokens']] for p in paragraphs]

def fetch(batch):
    try:
        results = ukb_senses(batch)
    except requests.HTTPError as error:
        if 'TS-117' not in error.response.text or len(batch) == 1:  # TS-117: text too long
            raise
        results = None
    if results is not None:
        return results
    middle = len(batch) // 2
    return fetch(batch[:middle]) + fetch(batch[middle:])

In [10]:
cache_path = Path('textserver_senses.json')
ukb = json.loads(cache_path.read_text()) if cache_path.is_file() else {}
missing = [s for s in unique_sentences if s not in ukb]
max_chars = 1500

batches, batch = [], []
for sentence in missing:
    if batch and sum(map(len, batch)) + len(sentence) > max_chars:
        batches.append(batch)
        batch = []
    batch.append(sentence)
if batch:
    batches.append(batch)

try:
    for batch in batches:
        ukb.update(zip(batch, fetch(batch)))
        cache_path.write_text(json.dumps(ukb, ensure_ascii=False, indent=1))
except requests.HTTPError as error:
    print('TextServer stopped:', error.response.text.strip())

ukb_complete = all(s in ukb for s in unique_sentences)
print(f'{sum(s in ukb for s in unique_sentences)} of {len(unique_sentences)} sentences processed with UKB')

if ukb_complete:
    ukb_sets, ukb_lemma_sets = {}, {}
    for sentence in unique_sentences:
        content = [t for t in ukb[sentence] if t[3] != 'punctuation']
        ukb_sets[sentence] = {t[4] for t in content if t[4] != 'N/A'}
        ukb_lemma_sets[sentence] = {t[4] if t[4] != 'N/A' else t[1].casefold() for t in content}
    print(f'{sum(t[4] != "N/A" for s in unique_sentences for t in ukb[s])} tokens got a sense')
    print(f'{sum(not s for s in ukb_sets.values())} sentences with no senses')
else:
    print('UKB is left out of the comparison until all sentences are processed (re-run this cell later).')

338 of 338 sentences processed with UKB
2072 tokens got a sense
2 sentences with no senses


In [11]:
def to_synset(sense):
    offset, pos = sense.split('-')
    return wn.synset_from_pos_and_offset(pos, int(offset)).name()

if ukb_complete:
    for a, b in pairs[:2]:
        print(a, sorted(map(to_synset, ukb_sets[a])), sep='\n  ')
        print(b, sorted(map(to_synset, ukb_sets[b])), sep='\n  ')
        print()

The leaders have now been given a new chance and let us hope they seize it.
  ['be.v.01', 'get.v.03', 'hold.v.03', 'hope.v.02', 'immediately.r.01', 'leader.n.01', 'new.a.06', 'opportunity.n.01', 'seize.v.01']
The leaders benefit aujourd' hui of a new luck and let's let them therefore seize it.
  ['benefit.v.02', 'leader.n.01', 'luck.n.02', 'new.a.06', 'permit.v.01', 'seize.v.01', 'therefore.r.01']

Amendment No 7 proposes certain changes in the references to paragraphs.
  ['amendment.n.02', 'certain.a.04', 'change.n.01', 'paragraph.n.01', 'propose.v.01', 'reference.n.08']
Amendment No 7 is proposing certain changes in the references to paragraphs.
  ['amendment.n.02', 'be.v.01', 'certain.a.04', 'change.n.01', 'paragraph.n.01', 'propose.v.01', 'reference.n.08']



## Comparison

The word correlation differs from session 2 (−0.450) because there we used `nltk.word_tokenize`, keeping capitalization and punctuation. Here we use lowercase words and regex tokenization, as in session 3, giving −0.485. The lemma correlation also matches session 3 (−0.483).

In [12]:
def distances(sets):
    return [jaccard_distance(sets[a], sets[b]) if sets[a] | sets[b] else 0.0 for a, b in pairs]

representations = {
    'Words': word_sets,
    'Lemmas': lemma_sets,
    'Lesk': sense_sets,
    'Lesk + lemmas': sense_lemma_sets,
}
if ukb_complete:
    representations |= {'UKB': ukb_sets, 'UKB + lemmas': ukb_lemma_sets}
dist = {name: distances(sets) for name, sets in representations.items()}

print(f"{'pair':<6} {'gold':<8}" + ''.join(f'{name:<15}' for name in dist))
for i in range(10):
    print(f"{i + 1:<6} {gs[i]:<8.3f}" + ''.join(f'{d[i]:<15.3f}' for d in dist.values()))

pair   gold    Words          Lemmas         Lesk           Lesk + lemmas  UKB            UKB + lemmas   
1      4.500   0.667          0.667          0.786          0.720          0.769          0.652          
2      5.000   0.231          0.083          0.500          0.357          0.143          0.083          
3      4.250   0.619          0.619          0.700          0.619          0.700          0.619          
4      4.500   0.417          0.417          0.500          0.417          0.600          0.545          
5      5.000   0.000          0.000          0.000          0.000          0.000          0.000          
6      5.000   0.632          0.632          0.833          0.700          0.750          0.700          
7      4.667   0.706          0.706          0.833          0.706          0.833          0.706          
8      5.000   0.250          0.250          0.000          0.000          0.200          0.250          
9      5.000   0.714          0.714          0

In [13]:
from statistics import correlation

print(f"{'Representation':<18} Pearson (distance)")
for name, d in dist.items():
    print(f"{name:<18} {correlation(gs, d):.6f}")

Representation     Pearson (distance)
Words              -0.485346
Lemmas             -0.482548
Lesk               -0.405574
Lesk + lemmas      -0.422369
UKB                -0.419182
UKB + lemmas       -0.418129


In [14]:
for i in (1, 3):
    a, b = pairs[i]
    print(f'pair {i + 1}: gold {gs[i]}, lemma distance {dist["Lemmas"][i]:.3f}, Lesk distance {dist["Lesk"][i]:.3f}')
    print('  only in 1:', sorted(sense_sets[a] - sense_sets[b]))
    print('  only in 2:', sorted(sense_sets[b] - sense_sets[a]))

pair 2: gold 5.0, lemma distance 0.083, Lesk distance 0.500
  only in 1: ['amendment.n.01', 'change.n.01']
  only in 2: ['amendment.n.02', 'be.v.12', 'change.n.05']
pair 4: gold 4.5, lemma distance 0.417, Lesk distance 0.500
  only in 1: ['meter.n.01', 'phosphorus.n.01', 'today.n.01']
  only in 2: []


**Lesk** and **UKB** use only the senses, so words without a sense (function words, numbers, names) are dropped; **+ lemmas** keeps those words as their lemma, so each sentence is represented by a mix of senses and lemmas.

## Justification

Words and lemmas give the strongest correlations, −0.485 and −0.483. UKB performs slightly better than Lesk when using only senses (−0.419 vs −0.406). Adding lemmas for words without a sense improves Lesk to −0.422, while UKB barely changes (−0.418). Both methods still perform worse than the word and lemma baselines.

For this dataset, we would choose words, because they give the strongest correlation and are simpler to compute. The difference with lemmas is very small, so either would be a reasonable choice. The correlations are negative because the gold scores measure similarity and we calculate Jaccard distance. Using 1 − distance would give the same correlations with positive signs.

Senses can help when different words express the same meaning, but WSD can also assign different meanings to words used in almost identical sentences. In pair 2, for example, Lesk chooses different senses for “amendment” and “changes”, although the gold similarity is 5. Its distance is 0.500, compared with 0.083 for lemmas. Keeping words without a sense reduces this to 0.357, but the incorrect senses still separate the sentences. The sense-only versions also drop words without a sense, so they lose information that the word and lemma baselines retain.